In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.config import *

In [0]:
cart_items = spark.read.table(f'{TABLE_PREFIX}.silver_cart_items')
products = spark.read.table(f'{TABLE_PREFIX}.dim_products')
users = spark.read.table(f'{TABLE_PREFIX}.dim_users')

In [0]:
cart_items = cart_items.alias('c')\
    .join(users.alias('u'), ((F.col('u.user_id') == F.col('c.user_id')) & (F.col('u.is_current') == True)))\
    .join(products.alias('p'), ((F.col('p.product_id') == F.col('c.product_id')) & (F.col('p.is_current') == True)))\
    .select(
        'c.cart_id',
        'p.product_sk',
        'u.user_sk',
        'c.quantity',
        F.col('c.line_total').alias('gross_amount'),
        F.col('c.line_discount_percentage').alias('discount_percentage'),
        F.col('c.line_discounted_total').alias('net_amount')
    )

In [0]:
cart_items.writeTo(f'{TABLE_PREFIX}.fact_cart_items').createOrReplace()